# Index setup and ingestion

**Notebook 2 of 5.** Creates three Azure AI Search indexes (`contoso-hr`,
`contoso-marketing`, `contoso-products`) with semantic configuration, then uploads
24 Contoso Corporation sample documents (8 per domain).

## Prerequisites

1. **Run `11-01-deploy-setup.ipynb`** - `.env` must contain `CONTOSO_SEARCH_ENDPOINT`,
   `CONTOSO_SEARCH_NAME`, and `CONTOSO_GATEWAY_KEY`.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - run `az login` before executing cells.

## Imports and configuration

In [1]:
import json
import subprocess
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)
from dotenv import load_dotenv

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

import os
SEARCH_ENDPOINT = os.environ['CONTOSO_SEARCH_ENDPOINT']
SEARCH_NAME     = os.environ['CONTOSO_SEARCH_NAME']

DOMAINS = ['hr', 'marketing', 'products']

print(f'Search endpoint : {SEARCH_ENDPOINT}')
print(f'Search name     : {SEARCH_NAME}')
print(f'Domains         : {DOMAINS}')

Search endpoint : https://contoso-search-n5d3ja.search.windows.net
Search name     : contoso-search-n5d3ja
Domains         : ['hr', 'marketing', 'products']


## Create clients

In [2]:
credential = DefaultAzureCredential()

index_client = SearchIndexClient(
    endpoint=SEARCH_ENDPOINT,
    credential=credential,
)

print('SearchIndexClient created')

SearchIndexClient created


## Define index schema

All three domain indexes share the same schema and a domain-specific semantic
configuration. Semantic search requires Standard SKU - confirmed during deployment.

| Field | Type | Key | Searchable | Filterable |
|-------|------|-----|------------|------------|
| `id` | `Edm.String` | ✓ | - | - |
| `content` | `Edm.String` | - | ✓ | - |
| `title` | `Edm.String` | - | ✓ | ✓ |
| `category` | `Edm.String` | - | ✓ | ✓ |
| `lastModified` | `Edm.DateTimeOffset` | - | - | ✓ |
| `url` | `Edm.String` | - | - | - |

In [3]:
def build_index(domain: str) -> SearchIndex:
    index_name    = f'contoso-{domain}'
    semantic_name = f'contoso-{domain}-semantic'

    fields = [
        SimpleField(
            name='id',
            type=SearchFieldDataType.String,
            key=True,
            filterable=False,
            sortable=False,
        ),
        SearchableField(
            name='content',
            type=SearchFieldDataType.String,
        ),
        SearchableField(
            name='title',
            type=SearchFieldDataType.String,
            filterable=True,
            sortable=True,
        ),
        SearchableField(
            name='category',
            type=SearchFieldDataType.String,
            filterable=True,
        ),
        SimpleField(
            name='lastModified',
            type=SearchFieldDataType.DateTimeOffset,
            filterable=True,
            sortable=True,
        ),
        SimpleField(
            name='url',
            type=SearchFieldDataType.String,
            filterable=False,
        ),
    ]

    semantic_config = SemanticConfiguration(
        name=semantic_name,
        prioritized_fields=SemanticPrioritizedFields(
            content_fields=[SemanticField(field_name='content')],
            title_field=SemanticField(field_name='title'),
            keywords_fields=[SemanticField(field_name='category')],
        ),
    )

    return SearchIndex(
        name=index_name,
        fields=fields,
        semantic_search=SemanticSearch(configurations=[semantic_config]),
    )


print('Index builder defined')

Index builder defined


## Create indexes

In [4]:
for domain in DOMAINS:
    idx = build_index(domain)
    result = index_client.create_or_update_index(idx)
    print(f'Index created/updated: {result.name}')

Index created/updated: contoso-hr
Index created/updated: contoso-marketing
Index created/updated: contoso-products


## Upload documents

In [5]:
lab_dir = Path.cwd()
sample_data_dir = lab_dir / 'sample_data'

for domain in DOMAINS:
    index_name = f'contoso-{domain}'
    data_file  = sample_data_dir / f'{domain}.json'

    docs = json.loads(data_file.read_text())

    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=index_name,
        credential=credential,
    )

    # Upload in a single batch (only 8 docs per domain)
    result = search_client.upload_documents(documents=docs)
    succeeded = sum(1 for r in result if r.succeeded)
    failed    = sum(1 for r in result if not r.succeeded)
    print(f'{index_name}: {succeeded} uploaded, {failed} failed')

contoso-hr: 8 uploaded, 0 failed
contoso-marketing: 8 uploaded, 0 failed
contoso-products: 8 uploaded, 0 failed


## Validate document counts

In [6]:
import time
time.sleep(3)  # brief pause for index freshness

for domain in DOMAINS:
    index_name = f'contoso-{domain}'
    search_client = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=index_name,
        credential=credential,
    )
    results = search_client.search('*', include_total_count=True)
    count = results.get_count()
    status = '✓' if count == 8 else f'⚠ expected 8'
    print(f'{index_name}: {count} documents {status}')

contoso-hr: 8 documents ✓
contoso-marketing: 8 documents ✓
contoso-products: 8 documents ✓


## Spot-check search results

In [7]:
spot_checks = [
    ('hr',        'remote work policy'),
    ('marketing', 'brand guidelines'),
    ('products',  'ContosoBook Pro'),
]

for domain, query in spot_checks:
    index_name = f'contoso-{domain}'
    sc = SearchClient(
        endpoint=SEARCH_ENDPOINT,
        index_name=index_name,
        credential=credential,
    )
    results = list(sc.search(query, top=1, select=['id', 'title', 'category']))
    if results:
        r = results[0]
        print(f'[{index_name}] "{query}" → {r["id"]}: {r["title"]} ({r["category"]})')
    else:
        print(f'[{index_name}] "{query}" → no results')

[contoso-hr] "remote work policy" → hr-001: Remote Work Policy (Work Arrangements)
[contoso-marketing] "brand guidelines" → mkt-002: Contoso Brand Guidelines (Brand)
[contoso-products] "ContosoBook Pro" → prd-008: ContosoEarBuds Pro (Audio)


## Done

Three Azure AI Search indexes are created and populated:

| Index | Documents | Semantic Config |
|-------|-----------|----------------|
| `contoso-hr` | 8 | `contoso-hr-semantic` |
| `contoso-marketing` | 8 | `contoso-marketing-semantic` |
| `contoso-products` | 8 | `contoso-products-semantic` |

**Next step:** run `11-03-knowledge-base-setup.ipynb` to create Knowledge Sources,
Knowledge Bases, and MCP connections.